# GraphLIME on RDF — a short live demo

Four steps, run live before the slides:

1. an RDF entity **has no features** — so there is nothing to explain;
2. we **build** the missing features out of the graph's own vocabulary;
3. on a graph whose answer we planted ourselves, the method **finds it**;
4. on the real datasets, the explanations **read as sentences**.

Nothing is retrained on AIFB/MUTAG: the models come from the committed
checkpoints. Rebuild this page with `just render` (~15 s).

In [1]:
from pathlib import Path

from graphlime_rdf.config import ExperimentConfig, SyntheticConfig
from graphlime_rdf.data.loader import load_rdf_graph
from graphlime_rdf.data.synthetic import generate_ground_truth_graph
from graphlime_rdf.explain.graphlime import Explanation, explain_node
from graphlime_rdf.features import build_features
from graphlime_rdf.pipeline import test_nodes_of
from graphlime_rdf.training import load_checkpoint, train_run

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLES_PER_DATASET = 4
TRIPLES_SHOWN = 5


def short(term: str) -> str:
    """Compact display form of a URI/term (literals keep their value)."""
    term = term.strip("<>")
    if term.startswith('"'):  # '"value"^^<datatype>' → '"value"'
        return term.split('"^^', 1)[0] + '"'
    for sep in ["#", "/"]:
        if sep in term:
            term = term.rsplit(sep, 1)[-1]
    return term or term

## 1. An RDF entity has no features

AIFB is a university's semantic portal — people, research groups,
publications. The task: *which group does this person belong to?*

In [2]:
graph = load_rdf_graph("aifb", root=REPO / "data")
NODE = test_nodes_of(graph)[0]  # the first test entity, not a cherry-picked one
me = short(graph.node_names[NODE])

print(
    f"AIFB: {graph.num_nodes} entities, {graph.edge_index.shape[1]} triples, "
    f"{graph.num_relations} predicates, {graph.num_classes} classes\n"
    f"worked example: node {NODE} — {me}\n"
)

src, dst = graph.edge_index
outgoing = (src == NODE).nonzero().flatten()
incoming = (dst == NODE).nonzero().flatten()
for e in outgoing[:TRIPLES_SHOWN]:
    predicate = short(graph.relation_names[int(graph.edge_type[int(e)])])
    print(f"  ({me})  --{predicate}-->  {short(graph.node_names[int(dst[int(e)])])}")
for e in incoming[:TRIPLES_SHOWN]:
    predicate = short(graph.relation_names[int(graph.edge_type[int(e)])])
    print(f"  {short(graph.node_names[int(src[int(e)])])}  --{predicate}-->  ({me})")

print(f"\n{len(outgoing)} outgoing + {len(incoming)} incoming triples, and not one attribute.")

AIFB: 8285 entities, 29043 triples, 45 predicates, 4 classes
worked example: node 5758 — id13instance

  (id13instance)  --fax-->  ""
  (id13instance)  --homepage-->  "http://www.aifb.uni-karlsruhe.de/WBS/sha"
  (id13instance)  --name-->  "Siegfried Handschuh"
  (id13instance)  --phone-->  ""
  (id13instance)  --photo-->  "http://www.aifb.uni-karlsruhe.de/Personen/Bilder/U1p2l3o4a5d13"
  id100instance  --isWorkedOnBy-->  (id13instance)
  id104instance  --isWorkedOnBy-->  (id13instance)
  id105instance  --isWorkedOnBy-->  (id13instance)
  id1instance  --isWorkedOnBy-->  (id13instance)
  id49instance  --isWorkedOnBy-->  (id13instance)

47 outgoing + 54 incoming triples, and not one attribute.


That is the whole difficulty. GraphLIME selects among the **columns of a
node feature matrix** — in a citation network every column is a word. Here
there is no matrix at all, and the usual R-GCN input is a one-hot entity id,
which would only ever let an explainer say *"entity #5758 mattered"*.

## 2. So we build the features from the graph

**Space A** counts predicates around the node — `out:<p>`, `in:<p>`.
**Space B** is finer, one column per predicate–object pair `out:<p>=<o>`.
Either way every column is already a term of the ontology, so anything the
selection returns is readable by construction.

In [3]:
config = ExperimentConfig.from_yaml(REPO / "configs" / "aifb.yaml")
x, vocabulary = build_features(graph, config.feature_space)
xb, vocab_b = build_features(
    graph, config.feature_space.model_copy(update={"kind": "predicate_object"})
)
print(f"space A (predicate counts):  {tuple(x.shape)}")
print(f"space B (predicate=object):  {tuple(xb.shape)}\n")

row = x[NODE]
print(f"the row of {me} in space A — {int((row > 0).sum())} non-zero of {x.shape[1]} columns:")
for j in row.nonzero().flatten():
    print(f"    {vocabulary.names[int(j)]:<55} {float(row[int(j)]):>6.0f}")

space A (predicate counts):  (8285, 90)
space B (predicate=object):  (8285, 1424)

the row of id13instance in space A — 11 non-zero of 90 columns:
    in:http://swrc.ontoware.org/ontology#author                 39
    in:http://swrc.ontoware.org/ontology#isWorkedOnBy           11
    in:http://swrc.ontoware.org/ontology#member                  4
    out:http://swrc.ontoware.org/ontology#fax                    1
    out:http://swrc.ontoware.org/ontology#homepage               1
    out:http://swrc.ontoware.org/ontology#name                   1
    out:http://swrc.ontoware.org/ontology#phone                  1
    out:http://swrc.ontoware.org/ontology#photo                  1
    out:http://swrc.ontoware.org/ontology#publication           39
    out:http://swrc.ontoware.org/ontology#worksAtProject         2
    out:http://www.w3.org/1999/02/22-rdf-syntax-ns#type          1


## 3. Does it actually find the right predicate?

On AIFB nobody knows the true reason for a label, so the claim is
unfalsifiable there. We therefore build a graph in which one predicate *is*
the class by construction, train on it, and check whether GraphLIME points
at exactly that predicate. (This replays the project's hard gate,
`tests/test_synthetic_gate.py`; without it no result is reported at all.)

In [4]:
syn_cfg = SyntheticConfig()
syn_graph = generate_ground_truth_graph(syn_cfg)
syn_config = ExperimentConfig.from_yaml(REPO / "configs" / "synthetic.yaml")
target = f"out:syn:p{syn_cfg.target_predicate}"
syn = train_run(syn_graph, syn_config, seed=0)

print(
    f"planted rule: class 1 ⟺ the entity has a {target} edge\n"
    f"({syn_cfg.num_entities} entities, {syn_cfg.num_predicates} predicates)\n"
    f"R-GCN test accuracy: {syn.manifest.final_test_accuracy:.4f}\n"
)

ranks = []
for node in test_nodes_of(syn_graph):
    out = explain_node(
        syn.model, node, syn.x, syn.edge_index, syn.edge_type,
        syn.vocabulary, syn_config.graphlime,
    )
    if isinstance(out, Explanation):
        names = [name for name, _ in out.top_features(len(out.beta))]
        ranks.append(names.index(target) + 1 if target in names else 10**6)

print(
    f"over {len(ranks)} test entities, {target} is ranked FIRST "
    f"{sum(r == 1 for r in ranks) / len(ranks):.1%} of the time."
)

planted rule: class 1 ⟺ the entity has a out:syn:p7 edge
(400 entities, 10 predicates)
R-GCN test accuracy: 1.0000



over 80 test entities, out:syn:p7 is ranked FIRST 100.0% of the time.


## 4. And on the real data, it speaks RDF

Same machinery, real entities, read out as *"predicted group X because of
predicate P toward Y"*. The model reads space A — its own vocabulary,
checked by hash against the checkpoint — while GraphLIME regresses on the
finer space B, so the explanation can name the term at the end of the edge.

In [5]:
def explain_examples(dataset: str) -> None:
    """Sentence-form explanations in feature space B: predicate *toward* term."""
    ckpt = load_checkpoint(REPO / "checkpoints" / f"{dataset}_best.pt")
    cfg = ckpt.manifest.resolved_config
    graph = load_rdf_graph(dataset, root=REPO / "data")
    x, vocabulary = build_features(graph, cfg.feature_space)
    assert vocabulary.hash == ckpt.manifest.vocabulary_hash
    space_b = cfg.feature_space.model_copy(update={"kind": "predicate_object"})
    xb, vocab_b = build_features(graph, space_b)
    # Qualitative display: gentler regularisation than the quantitative runs
    # (rho=0.1 in results/) so the sparser (p,o) features surface; stated here
    # openly — the recorded metrics never use this value.
    gl_cfg = cfg.graphlime.model_copy(update={"rho": 0.01})
    edge_index, edge_type = graph.doubled_edges()
    probs = ckpt.model.predict_proba(x, edge_index, edge_type)
    label_names = {i: name for name, i in ckpt.label_map.items()}

    shown = 0
    for node in test_nodes_of(graph):
        out = explain_node(
            ckpt.model, node, x, edge_index, edge_type, vocab_b, gl_cfg,
            interpretable_x=xb,
        )
        if not isinstance(out, Explanation):
            print(f"— node {node}: refused ({out.reason})\n")
            continue
        predicted = int(probs[node].argmax())
        correct = "correctly" if predicted == int(graph.labels[node]) else "INCORRECTLY"
        top = out.top_features(3)
        reasons = "; ".join(
            f"{short(name.split(':', 1)[1].split('=')[0])} "
            f"({'outgoing' if name.startswith('out:') else 'incoming'}, β={beta:.2f})"
            + (f" toward {short(name.split('=', 1)[1])}" if "=" in name else "")
            for name, beta in top
        ) or f"no feature selected at ρ={gl_cfg.rho} (locally uniform prediction)"
        print(
            f"{graph.dataset.upper()} node {node} — {short(graph.node_names[node])}\n"
            f"  {correct} predicted {short(label_names[predicted])} "
            f"(p={float(probs[node, predicted]):.2f}, "
            f"neighborhood={out.neighborhood_size})\n"
            f"  because of: {reasons}\n"
        )
        shown += 1
        if shown >= EXAMPLES_PER_DATASET:
            break

### AIFB — which research group does a person belong to?

In [6]:
explain_examples("aifb")

AIFB node 5758 — id13instance
  correctly predicted id3instance (p=1.00, neighborhood=1768)
  because of: publishes (incoming, β=0.06) toward id3instance; homepage (outgoing, β=0.05) toward "None"; type (outgoing, β=0.04) toward Person



AIFB node 5766 — id1851instance
  correctly predicted id1instance (p=1.00, neighborhood=1094)
  because of: member (incoming, β=0.05) toward id1instance; homepage (outgoing, β=0.04) toward "None"; author (incoming, β=0.04) toward id356instance



AIFB node 5769 — id1855instance
  correctly predicted id1instance (p=1.00, neighborhood=1094)
  because of: member (incoming, β=0.10) toward id2instance; member (incoming, β=0.06) toward id1instance; member (incoming, β=0.05) toward id4instance



AIFB node 5772 — id1860instance
  correctly predicted id1instance (p=1.00, neighborhood=1094)
  because of: fax (outgoing, β=0.05) toward ""; member (incoming, β=0.05) toward id2instance; isWorkedOnBy (incoming, β=0.04) toward id9instance



### MUTAG — is a compound mutagenic?

In [7]:
explain_examples("mutag")

MUTAG node 11481 — d103
  correctly predicted 0.0 (p=0.60, neighborhood=400)
  because of: cytogen_ca (outgoing, β=0.04) toward "false"; salmonella (outgoing, β=0.03) toward "false"; type (outgoing, β=0.02) toward DatatypeProperty

MUTAG node 11533 — d107
  correctly predicted 1.0 (p=1.00, neighborhood=721)
  because of: type (outgoing, β=0.20) toward Compound; type (outgoing, β=0.11) toward Chlorine-93; cytogen_ca (outgoing, β=0.10) toward "false"



MUTAG node 11676 — d108
  correctly predicted 0.0 (p=0.61, neighborhood=461)
  because of: cytogen_ca (outgoing, β=0.10) toward "false"; type (outgoing, β=0.05) toward Compound; salmonella (outgoing, β=0.04) toward "false"

MUTAG node 11792 — d112
  correctly predicted 1.0 (p=1.00, neighborhood=486)
  because of: cytogen_ca (outgoing, β=0.08) toward "false"; type (outgoing, β=0.08) toward Compound; mouse_lymph (outgoing, β=0.05) toward "false"



Each block is one test entity: the prediction with its confidence, the
neighbourhood GraphLIME sampled, and the three predicates with the largest
non-negative HSIC-Lasso weights β. The β are small in absolute terms because
the weight is spread over a sparse solution across hundreds of (p,o)
columns; what matters is which names come out on top. Refused nodes are
printed as refusals, never hidden.

---

That is the demo. How the method works, how faithful and how stable the
explanations are, and where they break down — that is the slide deck.